In [ ]:
# Cell 1: Load project inputs and select the UTM zone covering the largest share of the AOI.
import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
from pyproj import Geod
from rasterio.warp import Resampling, calculate_default_transform, reproject
from rasterio.windows import Window
from shapely.geometry import box, shape
from shapely.ops import unary_union

cwd = Path.cwd()
project_dir = cwd.parent if cwd.name.lower() == "notebooks" else cwd
input_dir = project_dir / "Input Data"
preprocessing_dir = project_dir / "Preprocessing"
output_dir = project_dir / "Output Data"
preprocessing_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

source_dem_path = input_dir / "sourceDEM.tif"
aoi_path = input_dir / "watershedAOI.geojson"
outlet_path = input_dir / "outlet.shp"
for required_path in (source_dem_path, aoi_path, outlet_path):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")

aoi_doc = json.loads(aoi_path.read_text(encoding="utf-8"))
aoi_features = aoi_doc.get("features", [])
if len(aoi_features) != 1:
    raise ValueError("watershedAOI.geojson must contain exactly one rectangle.")
aoi_geometry = shape(aoi_features[0]["geometry"])
if aoi_geometry.geom_type not in ("Polygon", "MultiPolygon") or not aoi_geometry.is_valid:
    raise ValueError("The AOI must be a valid rectangular polygon.")
min_lon, min_lat, max_lon, max_lat = aoi_geometry.bounds
if min_lat < -80 or max_lat > 84:
    raise ValueError("UTM is defined between 80 degrees south and 84 degrees north; redraw the AOI within that range.")
if max_lon - min_lon > 180:
    raise ValueError("The AOI crosses the antimeridian or spans over 180 degrees; choose a smaller UTM-compatible rectangle.")

geod = Geod(ellps="WGS84")
zone_geometries = {zone: [box(-180 + (zone - 1) * 6, -80, -180 + zone * 6, 84)] for zone in range(1, 61)}
# Apply the standard Norway and Svalbard UTM zone exceptions.
zone_geometries[31].append(box(6, 72, 9, 84))
zone_geometries[32].append(box(3, 56, 6, 64))
zone_geometries[33].extend((box(9, 72, 12, 84), box(18, 72, 21, 84)))
zone_geometries[34].append(box(18, 72, 24, 84))
zone_geometries[35].extend((box(21, 72, 24, 84), box(30, 72, 33, 84)))
zone_geometries[36].append(box(30, 72, 36, 84))
zone_geometries[37].append(box(33, 72, 36, 84))
zone_geometries[31][0] = zone_geometries[31][0].difference(box(3, 56, 6, 64))
zone_geometries[32][0] = zone_geometries[32][0].difference(box(6, 72, 12, 84))
zone_geometries[34][0] = zone_geometries[34][0].difference(box(18, 72, 24, 84))
zone_geometries[36][0] = zone_geometries[36][0].difference(box(30, 72, 36, 84))

zone_areas = {}
for zone, regions in zone_geometries.items():
    intersection = aoi_geometry.intersection(unary_union(regions))
    if intersection.is_empty:
        continue
    area_m2, _ = geod.geometry_area_perimeter(intersection)
    if area_m2:
        zone_areas[zone] = abs(area_m2)
if not zone_areas:
    raise ValueError("Could not determine a UTM zone for the selected AOI.")

utm_zone = max(zone_areas, key=zone_areas.get)
hemisphere_areas = {}
for north, south_box in ((True, box(-180, 0, 180, 84)), (False, box(-180, -80, 180, 0))):
    area_m2, _ = geod.geometry_area_perimeter(aoi_geometry.intersection(unary_union(zone_geometries[utm_zone]).intersection(south_box)))
    hemisphere_areas[north] = abs(area_m2)
northern_hemisphere = max(hemisphere_areas, key=hemisphere_areas.get)
utm_epsg = (32600 if northern_hemisphere else 32700) + utm_zone
dst_crs = f"EPSG:{utm_epsg}"
print(f"Selected UTM zone {utm_zone}{'N' if northern_hemisphere else 'S'} ({dst_crs}), based on the largest AOI area in that zone.")

In [ ]:
# Cell 2: Project the DEM and trim equal strips from all sides to make a NoData-free rectangle.
projected_dem_path = preprocessing_dir / "projectedDEM.tif"
watershed_dem_path = output_dir / "watershedDEM.tif"
dst_nodata = -9999.0

with rasterio.open(source_dem_path) as src:
    if src.crs is None:
        raise ValueError("The source DEM has no CRS; cannot select or apply a UTM projection.")
    transform, width, height = calculate_default_transform(
        src.crs,
        dst_crs,
        src.width,
        src.height,
        *src.bounds,
        resolution=30,
    )
    profile = src.profile.copy()
    profile.update(
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="float32",
        crs=dst_crs,
        transform=transform,
        nodata=dst_nodata,
        compress="lzw",
        tiled=True,
        blockxsize=256,
        blockysize=256,
    )
    with rasterio.open(projected_dem_path, "w", **profile) as dst:
        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=transform,
            dst_crs=dst_crs,
            dst_nodata=dst_nodata,
            resampling=Resampling.bilinear,
        )

with rasterio.open(projected_dem_path) as src:
    if min(src.width, src.height) < 3:
        raise ValueError("The projected DEM is too small to trim into a usable rectangle.")

    def is_nodata_free(row_start, col_start, row_stop, col_stop):
        for row in range(row_start, row_stop, 512):
            for col in range(col_start, col_stop, 512):
                window = Window(
                    col,
                    row,
                    min(512, col_stop - col),
                    min(512, row_stop - row),
                )
                values = src.read(1, window=window)
                valid = (src.read_masks(1, window=window) > 0) & np.isfinite(values)
                if src.nodata is not None:
                    valid &= values != src.nodata
                if not valid.all():
                    return False
        return True

    max_margin = (min(src.width, src.height) - 3) // 2
    if not is_nodata_free(max_margin, max_margin, src.height - max_margin, src.width - max_margin):
        raise ValueError(
            "NoData remains inside the largest centered rectangle after trimming its outer strips. "
            "Choose an AOI with a wider buffer or inspect the source DEM; interior NoData cannot be removed by edge trimming."
        )

    low_margin, high_margin = 0, max_margin
    while low_margin < high_margin:
        margin = (low_margin + high_margin) // 2
        if is_nodata_free(margin, margin, src.height - margin, src.width - margin):
            high_margin = margin
        else:
            low_margin = margin + 1

    strip_width = low_margin
    row_start = col_start = strip_width
    row_stop, col_stop = src.height - strip_width, src.width - strip_width
    crop_width, crop_height = col_stop - col_start, row_stop - row_start
    out_profile = src.profile.copy()
    out_profile.update(
        height=crop_height,
        width=crop_width,
        transform=src.window_transform(Window(col_start, row_start, crop_width, crop_height)),
        nodata=None,
        compress="lzw",
    )
    with rasterio.open(watershed_dem_path, "w", **out_profile) as dst:
        for row in range(row_start, row_stop, 512):
            for col in range(col_start, col_stop, 512):
                window = Window(
                    col,
                    row,
                    min(512, col_stop - col),
                    min(512, row_stop - row),
                )
                values = src.read(1, window=window)
                valid = (src.read_masks(1, window=window) > 0) & np.isfinite(values)
                if src.nodata is not None:
                    valid &= values != src.nodata
                if not valid.all():
                    raise ValueError("NoData was found during final DEM writing; the output was not completed.")
                dst.write(values, 1, window=Window(col - col_start, row - row_start, window.width, window.height))
with rasterio.open(watershed_dem_path) as src:
    print(f"NoData-free DEM saved: {watershed_dem_path}")
    print("Projection :", src.crs)
    print("Dimensions :", src.width, "x", src.height)
    print("Resolution :", src.res)
    print("Bounds     :", src.bounds)
    print("NoData tag :", src.nodata)

In [ ]:
# Cell 3: Fill DEM depressions before calculating drainage.
from whitebox.whitebox_tools import WhiteboxTools

wbt = WhiteboxTools()
wbt.set_working_dir(str(preprocessing_dir))
filled_dem_path = preprocessing_dir / "filledDEM.tif"
if filled_dem_path.exists():
    filled_dem_path.unlink()

status = wbt.fill_depressions(dem=str(watershed_dem_path), output=str(filled_dem_path))
if status != 0 or not filled_dem_path.exists():
    raise RuntimeError("WhiteboxTools failed to create the depression-filled DEM.")
print(f"Filled DEM saved: {filled_dem_path}")

In [ ]:
# Cell 4: Calculate the D8 flow pointer and accumulation from the pointer raster.
flow_pointer_path = preprocessing_dir / "flowPointer.tif"
flow_accumulation_path = preprocessing_dir / "flowAccumulation.tif"
for output_path in (flow_pointer_path, flow_accumulation_path):
    if output_path.exists():
        output_path.unlink()

status = wbt.d8_pointer(dem=str(filled_dem_path), output=str(flow_pointer_path))
if status != 0 or not flow_pointer_path.exists():
    raise RuntimeError("WhiteboxTools failed to create the D8 flow-pointer raster.")
status = wbt.d8_flow_accumulation(
    i=str(flow_pointer_path),
    output=str(flow_accumulation_path),
    out_type="cells",
    pntr=True,
)
if status != 0 or not flow_accumulation_path.exists():
    raise RuntimeError("WhiteboxTools failed to create the D8 flow-accumulation raster.")
print(f"D8 flow pointer: {flow_pointer_path}")
print(f"D8 flow accumulation: {flow_accumulation_path}")

In [ ]:
# Cell 5: Reproject the outlet and snap it to the highest-accumulation cell within the search radius.
import geopandas as gpd
from shapely.geometry import Point

outlet_utm_path = preprocessing_dir / "outlet_utm.shp"
snapped_outlet_path = preprocessing_dir / "outlet_snapped.shp"
outlet_gdf = gpd.read_file(outlet_path)
if len(outlet_gdf) != 1 or outlet_gdf.geometry.iloc[0].geom_type != "Point":
    raise ValueError("outlet.shp must contain exactly one point.")
if outlet_gdf.crs is None:
    raise ValueError("outlet.shp has no CRS; it must be saved in EPSG:4326.")
outlet_utm = outlet_gdf.to_crs(dst_crs)
with rasterio.open(watershed_dem_path) as src:
    bounds = src.bounds
    point = outlet_utm.geometry.iloc[0]
    if not (bounds.left <= point.x <= bounds.right and bounds.bottom <= point.y <= bounds.top):
        raise ValueError("The outlet falls outside the trimmed DEM. Redraw a larger AOI around both the outlet and the full catchment.")
    snap_distance_cells = 10
    snap_distance = snap_distance_cells * max(src.res)

outlet_utm.to_file(outlet_utm_path)
for old_file in preprocessing_dir.glob(f"{snapped_outlet_path.stem}.*"):
    old_file.unlink()
status = wbt.snap_pour_points(
    pour_pts=str(outlet_utm_path),
    flow_accum=str(flow_accumulation_path),
    output=str(snapped_outlet_path),
    snap_dist=snap_distance,
)
if status != 0 or not snapped_outlet_path.exists():
    raise RuntimeError("WhiteboxTools failed to snap the outlet to a nearby flow-accumulation cell.")
snapped_outlet = gpd.read_file(snapped_outlet_path)
if len(snapped_outlet) != 1 or snapped_outlet.geometry.iloc[0].geom_type != "Point":
    raise ValueError("The snapped outlet file does not contain exactly one point.")
movement = point.distance(snapped_outlet.geometry.iloc[0])
if movement > snap_distance + 1e-6:
    raise ValueError(f"The snapped outlet moved {movement:.1f} m, beyond the {snap_distance:.1f} m search radius.")
print(f"Outlet reprojected to {dst_crs}: {outlet_utm_path}")
print(f"Snapped outlet: {snapped_outlet_path}")
print(f"Snap search radius: {snap_distance:.0f} m ({snap_distance_cells} DEM cells); actual movement: {movement:.1f} m")

In [ ]:
# Cell 6: Delineate the snapped outlet catchment, save its boundary, and mask the DEM to that basin.
from rasterio.features import shapes
from shapely.ops import unary_union

watershed_raster_path = preprocessing_dir / "watershed.tif"
watershed_boundary_path = output_dir / "watershedBoundary.shp"
if watershed_raster_path.exists():
    watershed_raster_path.unlink()
status = wbt.watershed(
    d8_pntr=str(flow_pointer_path),
    pour_pts=str(snapped_outlet_path),
    output=str(watershed_raster_path),
)
if status != 0 or not watershed_raster_path.exists():
    raise RuntimeError("WhiteboxTools failed to delineate the watershed from the snapped outlet.")

with rasterio.open(watershed_raster_path) as src:
    watershed = src.read(1)
    basin_mask = np.isfinite(watershed) & (watershed > 0)
    if src.nodata is not None:
        basin_mask &= watershed != src.nodata
    if not basin_mask.any():
        raise ValueError("No watershed cells were delineated. Check the snapped outlet and D8 flow pointer.")
    basin_rows, basin_cols = np.where(basin_mask)
    row_start, row_stop = int(basin_rows.min()), int(basin_rows.max()) + 1
    col_start, col_stop = int(basin_cols.min()), int(basin_cols.max()) + 1
    watershed_features = [
        shape(geometry)
        for geometry, value in shapes(watershed, mask=basin_mask, transform=src.transform)
        if value > 0
    ]
    watershed_geometry = unary_union(watershed_features)
    watershed_crs = src.crs
    basin_id = int(np.max(watershed[basin_mask]))

watershed_gdf = gpd.GeoDataFrame(
    {"name": ["watershed"], "ws_id": [basin_id]},
    geometry=[watershed_geometry],
    crs=watershed_crs,
)
watershed_gdf["area_km2"] = watershed_gdf.geometry.area / 1_000_000
for old_file in output_dir.glob(f"{watershed_boundary_path.stem}.*"):
    old_file.unlink()
watershed_gdf.to_file(watershed_boundary_path)

with rasterio.open(watershed_dem_path) as src:
    if src.crs != watershed_crs or src.width != basin_mask.shape[1] or src.height != basin_mask.shape[0]:
        raise ValueError("The watershed raster does not align with the projected DEM.")
    crop = Window(
        col_start,
        row_start,
        col_stop - col_start,
        row_stop - row_start,
    )
    clipped_dem = src.read(1, window=crop).astype("float32", copy=False)
    clipped_mask = basin_mask[row_start:row_stop, col_start:col_stop]
    clipped_dem[~clipped_mask] = -9999.0
    clipped_profile = src.profile.copy()
    clipped_profile.update(
        height=clipped_dem.shape[0],
        width=clipped_dem.shape[1],
        transform=src.window_transform(crop),
        dtype="float32",
        nodata=-9999.0,
        compress="lzw",
    )

with rasterio.open(watershed_dem_path, "w", **clipped_profile) as dst:
    dst.write(clipped_dem, 1)

print(f"Delineated watershed boundary saved: {watershed_boundary_path}")
print(f"DEM clipped to this outlet's watershed: {watershed_dem_path}")
print(watershed_gdf[["ws_id", "area_km2"]])

In [ ]:
# Cell 7: Plot the rectangular DEM with watershed boundary, then the watershed-only DEM.
import matplotlib.pyplot as plt

rectangular_png_path = output_dir / "watershedDEM_with_boundary.png"
with rasterio.open(projected_dem_path) as src:
    rectangular_scale = min(1.0, 2400 / max(src.width, src.height))
    rectangular_height = max(1, round(src.height * rectangular_scale))
    rectangular_width = max(1, round(src.width * rectangular_scale))
    rectangular_dem = src.read(
        1,
        out_shape=(rectangular_height, rectangular_width),
        resampling=Resampling.average,
    )
    rectangular_mask = src.read_masks(
        1,
        out_shape=(rectangular_height, rectangular_width),
        resampling=Resampling.nearest,
    ) == 0
    rectangular_bounds = src.bounds
    rectangular_extent = [
        rectangular_bounds.left,
        rectangular_bounds.right,
        rectangular_bounds.bottom,
        rectangular_bounds.top,
    ]

if rectangular_mask.all():
    raise ValueError("The rectangular DEM contains no valid pixels to plot.")
rectangular_dem = np.ma.array(rectangular_dem, mask=rectangular_mask)
rectangular_low, rectangular_high = np.percentile(rectangular_dem.compressed(), [2, 98])
if rectangular_low == rectangular_high:
    rectangular_low = float(rectangular_dem.min())
    rectangular_high = float(rectangular_dem.max()) + 1

rectangular_cmap = plt.get_cmap("terrain").copy()
rectangular_cmap.set_bad((1, 1, 1, 0))
rectangular_fig, rectangular_ax = plt.subplots(figsize=(11, 9), constrained_layout=True)
rectangular_ax.set_facecolor("#e9eef2")
rectangular_image = rectangular_ax.imshow(
    rectangular_dem,
    extent=rectangular_extent,
    origin="upper",
    cmap=rectangular_cmap,
    vmin=float(rectangular_low),
    vmax=float(rectangular_high),
    interpolation="bilinear",
)
watershed_gdf.boundary.plot(
    ax=rectangular_ax,
    color="#252525",
    linewidth=1.0,
    label="Watershed boundary",
    zorder=4,
)
rectangular_outlet = gpd.read_file(snapped_outlet_path).to_crs(watershed_gdf.crs).geometry.iloc[0]
rectangular_ax.scatter(
    rectangular_outlet.x,
    rectangular_outlet.y,
    s=58,
    marker="o",
    color="#d73027",
    edgecolors="white",
    linewidths=1.2,
    label="Snapped outlet",
    zorder=6,
)
rectangular_ax.set_title("Rectangular DEM with Delineated Watershed", fontsize=15, weight="bold")
rectangular_ax.set_xlabel("Easting (m)")
rectangular_ax.set_ylabel("Northing (m)")
rectangular_ax.set_aspect("equal")
rectangular_map_width = rectangular_bounds.right - rectangular_bounds.left
rectangular_map_height = rectangular_bounds.top - rectangular_bounds.bottom
rectangular_padding = 0.035
rectangular_ax.set_xlim(
    rectangular_bounds.left - rectangular_map_width * rectangular_padding,
    rectangular_bounds.right + rectangular_map_width * rectangular_padding,
)
rectangular_ax.set_ylim(
    rectangular_bounds.bottom - rectangular_map_height * rectangular_padding,
    rectangular_bounds.top + rectangular_map_height * rectangular_padding,
)
rectangular_ax.annotate(
    "",
    xy=(0.95, 0.97),
    xytext=(0.95, 0.87),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "-|>", "color": "#202020", "linewidth": 1.8},
    zorder=8,
)
rectangular_ax.text(
    0.95,
    0.985,
    "N",
    transform=rectangular_ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=12,
    fontweight="bold",
    color="#202020",
)
rectangular_scale_target = rectangular_map_width * 0.18
rectangular_scale_power = 10 ** np.floor(np.log10(rectangular_scale_target))
rectangular_scale_length = next(
    factor * rectangular_scale_power
    for factor in (5, 2, 1)
    if factor * rectangular_scale_power <= rectangular_scale_target
)
rectangular_scale_x = rectangular_bounds.left + rectangular_map_width * 0.045
rectangular_scale_y = rectangular_bounds.bottom + rectangular_map_height * 0.045
rectangular_scale_label = (
    f"{rectangular_scale_length / 1000:g} km"
    if rectangular_scale_length >= 1000
    else f"{rectangular_scale_length:g} m"
)
rectangular_ax.plot(
    [rectangular_scale_x, rectangular_scale_x + rectangular_scale_length],
    [rectangular_scale_y, rectangular_scale_y],
    color="#202020",
    linewidth=3,
    solid_capstyle="butt",
    zorder=8,
)
rectangular_ax.plot(
    [rectangular_scale_x, rectangular_scale_x],
    [rectangular_scale_y - rectangular_map_height * 0.006, rectangular_scale_y + rectangular_map_height * 0.006],
    color="#202020",
    linewidth=1.5,
    zorder=8,
)
rectangular_ax.plot(
    [rectangular_scale_x + rectangular_scale_length, rectangular_scale_x + rectangular_scale_length],
    [rectangular_scale_y - rectangular_map_height * 0.006, rectangular_scale_y + rectangular_map_height * 0.006],
    color="#202020",
    linewidth=1.5,
    zorder=8,
)
rectangular_ax.text(
    rectangular_scale_x + rectangular_scale_length / 2,
    rectangular_scale_y + rectangular_map_height * 0.012,
    rectangular_scale_label,
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
    color="#202020",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 2},
    zorder=9,
)
rectangular_ax.legend(loc="upper left", frameon=True, framealpha=0.9)
rectangular_colorbar = rectangular_fig.colorbar(rectangular_image, ax=rectangular_ax, shrink=0.82, pad=0.03)
rectangular_colorbar.set_label("Elevation (m)")
rectangular_fig.savefig(rectangular_png_path, dpi=240, bbox_inches="tight", facecolor="white")
print(f"Rectangular DEM with watershed boundary saved: {rectangular_png_path}")
plt.show()

png_path = output_dir / "watershedDEM.png"
with rasterio.open(watershed_dem_path) as src:
    scale = min(1.0, 2400 / max(src.width, src.height))
    out_height = max(1, round(src.height * scale))
    out_width = max(1, round(src.width * scale))
    dem_preview = src.read(
        1,
        out_shape=(out_height, out_width),
        resampling=Resampling.average,
    )
    preview_mask = src.read_masks(
        1,
        out_shape=(out_height, out_width),
        resampling=Resampling.nearest,
    ) == 0
    bounds = src.bounds
    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

if preview_mask.all():
    raise ValueError("The clipped watershed DEM contains no valid pixels to plot.")
dem_preview = np.ma.array(dem_preview, mask=preview_mask)
low, high = np.percentile(dem_preview.compressed(), [2, 98])
if low == high:
    low, high = float(dem_preview.min()), float(dem_preview.max()) + 1

cmap = plt.get_cmap("terrain").copy()
cmap.set_bad((1, 1, 1, 0))
fig, ax = plt.subplots(figsize=(11, 9), constrained_layout=True)
ax.set_facecolor("#e9eef2")
image = ax.imshow(
    dem_preview,
    extent=extent,
    origin="upper",
    cmap=cmap,
    vmin=float(low),
    vmax=float(high),
    interpolation="bilinear",
)
watershed_gdf.boundary.plot(ax=ax, color="#252525", linewidth=0.8, zorder=4)

snapped_outlet_for_plot = gpd.read_file(snapped_outlet_path).to_crs(watershed_gdf.crs)
outlet_point = snapped_outlet_for_plot.geometry.iloc[0]
ax.scatter(
    outlet_point.x,
    outlet_point.y,
    s=58,
    marker="o",
    color="#d73027",
    edgecolors="white",
    linewidths=1.2,
    label="Snapped outlet",
    zorder=6,
)

ax.set_title("Watershed DEM", fontsize=16, weight="bold")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_aspect("equal")

map_width = bounds.right - bounds.left
map_height = bounds.top - bounds.bottom
padding = 0.055
ax.set_xlim(bounds.left - map_width * padding, bounds.right + map_width * padding)
ax.set_ylim(bounds.bottom - map_height * padding, bounds.top + map_height * padding)

ax.annotate(
    "",
    xy=(0.94, 0.97),
    xytext=(0.94, 0.87),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "-|>", "color": "#202020", "linewidth": 1.8},
    zorder=8,
)
ax.text(
    0.94,
    0.985,
    "N",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=12,
    fontweight="bold",
    color="#202020",
)

scale_target = map_width * 0.18
scale_power = 10 ** np.floor(np.log10(scale_target))
scale_length = next(
    factor * scale_power
    for factor in (5, 2, 1)
    if factor * scale_power <= scale_target
)
scale_x = bounds.left + map_width * 0.055
scale_y = bounds.bottom + map_height * 0.055
scale_label = f"{scale_length / 1000:g} km" if scale_length >= 1000 else f"{scale_length:g} m"
ax.plot(
    [scale_x, scale_x + scale_length],
    [scale_y, scale_y],
    color="#202020",
    linewidth=3,
    solid_capstyle="butt",
    zorder=8,
)
ax.plot(
    [scale_x, scale_x],
    [scale_y - map_height * 0.008, scale_y + map_height * 0.008],
    color="#202020",
    linewidth=1.5,
    zorder=8,
)
ax.plot(
    [scale_x + scale_length, scale_x + scale_length],
    [scale_y - map_height * 0.008, scale_y + map_height * 0.008],
    color="#202020",
    linewidth=1.5,
    zorder=8,
)
ax.text(
    scale_x + scale_length / 2,
    scale_y + map_height * 0.014,
    scale_label,
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
    color="#202020",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 2},
    zorder=9,
)

ax.legend(loc="upper left", frameon=True, framealpha=0.9)
colorbar = fig.colorbar(image, ax=ax, shrink=0.82, pad=0.03)
colorbar.set_label("Elevation (m)")
fig.savefig(png_path, dpi=240, bbox_inches="tight", facecolor="white")
print(f"Watershed-only DEM colour-ramp image saved: {png_path}")
plt.show()


In [ ]:
# Cell 8: List the final watershed deliverables and supporting files.
print("Final watershed deliverables:")
for path in (watershed_dem_path, watershed_boundary_path, png_path):
    if not path.exists():
        raise FileNotFoundError(f"Expected deliverable was not created: {path}")
    print(f"- {path} ({path.stat().st_size / (1024 ** 2):.2f} MB)")
print("Supporting boundary files:")
for path in sorted(output_dir.glob(f"{watershed_boundary_path.stem}.*")):
    print(f"- {path.name}")
print("Supporting processing files:")
for path in sorted(preprocessing_dir.iterdir()):
    if path.is_file():
        print(f"- {path.name}")